In [1]:
# Cell 1: Load the Heart Dataset

file_path = "../../datasets/heart.csv"

with open(file_path, "r", encoding="utf-8-sig") as file:
    lines = file.read().splitlines()

headers = lines[0].split(",")

data = []

for line in lines[1:]:
    if line.strip():
        row = line.split(",")

        if len(row) == len(headers):
            data.append(row)

print("Dataset loaded successfully!")
print("Total rows:", len(data))
print("Total columns:", len(headers))
print("Columns:", headers)

Dataset loaded successfully!
Total rows: 1025
Total columns: 14
Columns: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']


In [2]:
# Cell 2: Manual missing value detection

missing_markers = ["", "na", "n/a", "null", "none", "nan"]

def is_missing(value):
    return value.strip().casefold() in missing_markers


def count_missing_values(data, headers):
    missing_counts = {}

    for column_index, column_name in enumerate(headers):
        count = 0

        for row in data:
            if is_missing(row[column_index]):
                count += 1

        missing_counts[column_name] = count

    return missing_counts


missing_counts = count_missing_values(data, headers)

for column, count in missing_counts.items():
    print(f"{column}: {count} missing values")

total_missing = 0

for count in missing_counts.values():
    total_missing += count

print("\nTotal missing values:", total_missing)

age: 0 missing values
sex: 0 missing values
cp: 0 missing values
trestbps: 0 missing values
chol: 0 missing values
fbs: 0 missing values
restecg: 0 missing values
thalach: 0 missing values
exang: 0 missing values
oldpeak: 0 missing values
slope: 0 missing values
ca: 0 missing values
thal: 0 missing values
target: 0 missing values

Total missing values: 0


In [3]:
# Cell 3: Calculate missing percentage manually

total_cells = len(data) * len(headers)

if total_cells > 0:
    missing_percentage = total_missing * 100 / total_cells
else:
    missing_percentage = 0

print("Total cells:", total_cells)
print("Missing cells:", total_missing)
print("Missing percentage:", round(missing_percentage, 2), "%")

Total cells: 14350
Missing cells: 0
Missing percentage: 0.0 %


In [4]:
# Cell 4: Extract valid numerical values

def get_numeric_values(data, column_index):
    values = []

    for row in data:
        value = row[column_index].strip()

        if is_missing(value):
            continue

        try:
            values.append(float(value))
        except ValueError:
            continue

    return values


numeric_columns = []

for column_index, column_name in enumerate(headers):
    values = get_numeric_values(data, column_index)

    if len(values) > 0:
        numeric_columns.append(column_name)

print("Numerical columns:", numeric_columns)

Numerical columns: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']


In [5]:
# Cell 5: Manual statistical functions

def manual_mean(values):
    if not values:
        return None

    total = 0.0
    count = 0

    for value in values:
        total += value
        count += 1

    return total / count


def manual_sort(values):
    result = []

    for value in values:
        position = 0

        while position < len(result) and result[position] <= value:
            position += 1

        result.insert(position, value)

    return result


def manual_median(values):
    if not values:
        return None

    ordered = manual_sort(values)
    middle = len(ordered) // 2

    if len(ordered) % 2 == 1:
        return ordered[middle]

    return (ordered[middle - 1] + ordered[middle]) / 2


def manual_mode(values):
    if not values:
        return None

    frequency = {}

    for value in values:
        frequency[value] = frequency.get(value, 0) + 1

    highest_frequency = 0

    for value in frequency:
        if frequency[value] > highest_frequency:
            highest_frequency = frequency[value]

    # If every value occurs equally often, no unique mode exists.
    if highest_frequency == 1:
        return None

    for value in frequency:
        if frequency[value] == highest_frequency:
            return value

In [6]:
# Cell 6: Imputation methods

def impute_mean(data, headers, column_name):
    column_index = headers.index(column_name)
    values = get_numeric_values(data, column_index)
    replacement = manual_mean(values)

    result = [row.copy() for row in data]

    if replacement is None:
        return result

    for row in result:
        if is_missing(row[column_index]):
            row[column_index] = str(replacement)

    return result


def impute_median(data, headers, column_name):
    column_index = headers.index(column_name)
    values = get_numeric_values(data, column_index)
    replacement = manual_median(values)

    result = [row.copy() for row in data]

    if replacement is None:
        return result

    for row in result:
        if is_missing(row[column_index]):
            row[column_index] = str(replacement)

    return result


def impute_mode(data, headers, column_name):
    column_index = headers.index(column_name)

    result = [row.copy() for row in data]
    frequencies = {}

    for row in result:
        value = row[column_index].strip()

        if not is_missing(value):
            frequencies[value] = frequencies.get(value, 0) + 1

    if not frequencies:
        return result

    highest_frequency = 0
    replacement = None

    for value in frequencies:
        if frequencies[value] > highest_frequency:
            highest_frequency = frequencies[value]
            replacement = value

    for row in result:
        if is_missing(row[column_index]):
            row[column_index] = replacement

    return result


def impute_forward_fill(data, headers, column_name):
    column_index = headers.index(column_name)
    result = [row.copy() for row in data]
    previous_value = None

    for row in result:
        value = row[column_index].strip()

        if is_missing(value):
            if previous_value is not None:
                row[column_index] = previous_value
        else:
            previous_value = row[column_index]

    return result

In [7]:
# Cell 7: Compare imputation methods

column_name = "chol"
column_index = headers.index(column_name)

print("Missing values in chol:", missing_counts[column_name])

mean_data = impute_mean(data, headers, column_name)
median_data = impute_median(data, headers, column_name)
mode_data = impute_mode(data, headers, column_name)
forward_data = impute_forward_fill(data, headers, column_name)

print("Mean imputation completed.")
print("Median imputation completed.")
print("Mode imputation completed.")
print("Forward fill completed.")

Missing values in chol: 0
Mean imputation completed.
Median imputation completed.
Mode imputation completed.
Forward fill completed.


In [8]:
# Cell 8: Controlled demonstration on a copy

test_data = [row.copy() for row in data]

column_name = "chol"
column_index = headers.index(column_name)

# Find a valid value and replace it only in the test copy.
test_index = None

for i in range(len(test_data)):
    if not is_missing(test_data[i][column_index]):
        test_index = i
        break

if test_index is not None:
    original_value = test_data[test_index][column_index]
    test_data[test_index][column_index] = ""

    print("Created one test missing value in chol.")
    print("Original value:", original_value)
    print(
        "Test missing count:",
        count_missing_values(test_data, headers)[column_name]
    )

    mean_test = impute_mean(test_data, headers, column_name)
    median_test = impute_median(test_data, headers, column_name)
    mode_test = impute_mode(test_data, headers, column_name)
    forward_test = impute_forward_fill(test_data, headers, column_name)

    print("Mean replacement:", mean_test[test_index][column_index])
    print("Median replacement:", median_test[test_index][column_index])
    print("Mode replacement:", mode_test[test_index][column_index])
    print("Forward-fill replacement:", forward_test[test_index][column_index])
else:
    print("No valid cholesterol value available for testing.")

Created one test missing value in chol.
Original value: 212
Test missing count: 1
Mean replacement: 246.033203125
Median replacement: 240.0
Mode replacement: 204
Forward-fill replacement: 


Conclusion
- Mean imputation is suitable for numerical data without strong outliers.
- Median imputation is more robust when numerical data contains outliers or skewness.
- Mode imputation is suitable for categorical values and can also be used for discrete numerical data.
- Forward fill is appropriate for ordered data where the previous observation is meaningful.
- The original dataset must be preserved before applying imputation.
- Imputation should only be applied after identifying missing values and understanding the data.